# 갑상선암 검진 정책 변화 분석 (학생용 1-클릭 Colab)

이 노트북은 **저장소 경로와 파일명이 이미 고정**되어 있어, **학생들이 Colab에서 코드 셀의 ▶ 버튼 1번만 누르면** 분석이 시작되도록 만든 버전입니다.

출력물:
- 전체/남녀 연령표준화발생률 추세
- 2012=100 충격지수
- 버터플라이 3단 비교
- 연령 heatmap
- 감소율 / rebound 워터폴
- 버블차트
- ITS 분석
- 자동 해석 요약


In [ ]:
import os, sys, subprocess, urllib.request, shutil
from urllib.parse import quote
from IPython.display import display, Markdown

print("환경 설정 중...")
subprocess.run(["apt-get", "-qq", "install", "fonts-nanum"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "numpy", "matplotlib", "statsmodels", "openpyxl", "scipy"], check=True)

# =========================
# 저장소 고정 설정
# =========================
GITHUB_USER = "runmc3812"
REPO_NAME = "Healthcare_Informatics_Integration-2026-"
BRANCH = "main"
BASE_PATH = "1조"

DATA_DIR = "data"
SCRIPT_DIR = "src"
NOTEBOOK_DIR = "notebooks"

MAIN_FILE = "국립암센터_24개종 암발생률_20260120.csv"
AGE_FILE = "24개_암종_성_연령_5세_별_암발생자수__발생률_20260324142549.csv"
SCRIPT_FILE = "thyroid_colab_analysis.py"

OUTDIR = "/content/thyroid_analysis_outputs"

def gh_raw_url(*parts):
    encoded = "/".join(quote(str(p), safe="") for p in parts)
    return f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{encoded}"

main_source = gh_raw_url(BASE_PATH, DATA_DIR, MAIN_FILE)
age_source = gh_raw_url(BASE_PATH, DATA_DIR, AGE_FILE)
script_source = gh_raw_url(BASE_PATH, SCRIPT_DIR, SCRIPT_FILE)

print("main_source =", main_source)
print("age_source =", age_source)
print("script_source =", script_source)

# 스크립트 다운로드
local_script = "/content/thyroid_colab_analysis.py"
urllib.request.urlretrieve(script_source, local_script)
sys.path.append("/content")

from thyroid_colab_analysis import run_analysis

print("\n분석 실행 중...")
results = run_analysis(main_source=main_source, age_source=age_source, outdir=OUTDIR)

display(Markdown("## 핵심 요약표"))
display(results["summary_change"])

display(Markdown("## 2012→2015 감소폭 상위 10개 연령군"))
display(results["age_change_total"].sort_values("2012→2015 변화율(%)").head(10))

display(Markdown("## 2015→2023 rebound 상위 10개 연령군"))
display(results["age_change_total"].sort_values("2015→2023 변화율(%)", ascending=False).head(10))

summary_path = os.path.join(OUTDIR, "auto_summary.txt")
if os.path.exists(summary_path):
    with open(summary_path, "r", encoding="utf-8") as f:
        print("\n" + f.read())

zip_path = shutil.make_archive("/content/thyroid_analysis_outputs", "zip", OUTDIR)
print("\n압축 완료:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("자동 다운로드는 Colab에서만 동작합니다:", e)

print("\n완료: 학생들은 이 노트북에서 이 코드 셀의 ▶ 버튼만 누르면 됩니다.")